# 05: Adaptive Attacker Defense Test (Adaptive Attacker)

**Cold-Start Social-Graph Leakage: Strengthening the Threat Model**

The article's defense evaluation (04) measures defenses against a **fixed** attacker. A real attacker knows the defense and gives the **best-response**. This notebook tests the following thesis by measuring it:

> Defenses such as output noise, pool expansion and diversification appear robust against a **fixed** attacker, but an **adapted** attacker (query-aggregation for noise/pool, budget-expansion for diversification) recovers the leakage to a level nearly as vulnerable as undefended. **Rate-limiting** is the only complement that survives, because it directly limits the growth of recovery with query budget.

**Design fidelity:** generator + LightGCN + BPR + normalized embedding are **identical** to 02/02b. Attacker-value is again the cold-start context fallback (h=0): score = sim(e_target, e_c). Defenses are applied as a single re-ranking/shuffling layer on top of this score vector; the fixed and adaptive attacker are measured against the same layer.

## 0. Kurulum

In [ ]:
!pip install -q networkx numpy pandas scipy torch matplotlib

In [ ]:
# --- repository-root anchor (added when notebooks moved into notebooks/) ---
# Notebooks live in notebooks/ but read and write cold_start_leakage/ at the
# repository root, so resolve that root explicitly instead of relying on cwd.
import os as _os_anchor
_here = _os_anchor.getcwd()
if _os_anchor.path.basename(_here) == 'notebooks':
    _os_anchor.chdir(_os_anchor.path.dirname(_here))
# --- end anchor ---
import os, json, time, math
import numpy as np, pandas as pd, networkx as nx
import torch, torch.nn as nn
from scipy.stats import norm
from scipy.optimize import brentq
def _in_colab():
    try:
        import google.colab; return True
    except Exception: return False
if _in_colab():
    from google.colab import drive; drive.mount('/content/drive')
    ROOT='/content/drive/MyDrive/cold_start_leakage'
else:
    ROOT=os.path.abspath('./cold_start_leakage')
OUT_DIR=os.path.join(ROOT,'adaptive'); os.makedirs(OUT_DIR,exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Output:",OUT_DIR,"| torch",torch.__version__,"| device:",DEVICE)

## 1. Ayarlar

In [ ]:
MODE="quick"            # "quick" | "full"
BASELINE=dict(p_in=0.06,p_out=0.006,recip=0.49,homo=0.70,priv=0.30)
K=20
SIGMAS=(0.1,0.2,0.4)                    # output-noise defense
EPS_STEPS=(20000,2000,200)              # real DP defense (Reviewer 3, item 5): per-query epsilon
DP_DELTA=1e-5; DP_CLIP=1.0              # same definition and rationale as notebook 05
Qs=(1,2,5,10,20,40)                     # attacker query budget / rate-limit Q_max
LAMS=(0.1,0.5,0.9)                      # diversification (lower lam = more aggressive)
DIVBUDG=(1,2,3,5)                       # adaptive diversification: K'=b*K
POOLS=(0.3,0.5,0.7)                     # pool dilution ratio p
POOL_M=120                              # MMR candidate-pool size
# nominal operating points reported in the paper:
NOM=dict(sig=0.2,p=0.5,lam=0.1,eps=2000)
if MODE=="quick":
    SEEDS=[0,1,2]; GRAPHS=["sbm"]; EPOCHS=30
else:
    SEEDS=list(range(10)); GRAPHS=["sbm","lfr"]; EPOCHS=45
N_NODES=700
print(f"MODE={MODE} | {len(SEEDS)} seed x {len(GRAPHS)} graph | device={DEVICE}")

## 2. Generator and model, identical to notebook 03

In [ ]:
class DS:
    def __init__(s,G,priv,comm,nl,X): s.G,s.private,s.community,s.node_list,s.features=G,priv,comm,nl,X
    def neigh(s,t): return set(s.G.predecessors(t))|set(s.G.successors(t))
def _orient(Gu,r,rng):
    D=nx.DiGraph(); D.add_nodes_from(Gu.nodes())
    for u,v in Gu.edges():
        if u==v: continue
        if rng.random()<r: D.add_edge(u,v); D.add_edge(v,u)
        else: (D.add_edge(u,v) if rng.random()<0.5 else D.add_edge(v,u))
    return D
def _feat(n,nc,comm,homo,fd,rng):
    mu=rng.normal(size=(nc,fd)); X=np.zeros((n,fd))
    for i in range(n): X[i]=homo*mu[comm[i]]+(1-homo)*rng.normal(size=fd)
    return ((X-X.mean(0))/(X.std(0)+1e-9)).astype(np.float32)
def _lab(n,priv,rng):
    pr={i:False for i in range(n)}
    for i in rng.choice(n,int(round(priv*n)),replace=False): pr[i]=True
    return pr
def make_sbm(n=700,nc=7,p_in=0.06,p_out=0.006,recip=0.49,homo=0.70,priv=0.30,fd=16,seed=0):
    rng=np.random.default_rng(seed); sizes=[n//nc]*nc
    for i in range(n-sum(sizes)): sizes[i]+=1
    P=np.full((nc,nc),p_out); np.fill_diagonal(P,p_in)
    Gu=nx.stochastic_block_model(sizes,P,seed=int(seed))
    comm={i:Gu.nodes[i]['block'] for i in Gu.nodes()}
    return DS(_orient(Gu,recip,rng),_lab(n,priv,rng),comm,list(range(n)),_feat(n,nc,comm,homo,fd,rng))
def make_lfr(n=700,recip=0.49,homo=0.70,priv=0.30,fd=16,seed=0,mu=0.1,**_):
    rng=np.random.default_rng(seed); Gu=None
    for off in range(8):
        try:
            Gu=nx.LFR_benchmark_graph(n=n,tau1=3,tau2=1.5,mu=mu,average_degree=10,min_community=30,seed=int(seed)+off,max_iters=500); break
        except Exception: continue
    if Gu is None: return None
    Gu=nx.Graph(Gu); Gu.remove_edges_from(nx.selfloop_edges(Gu)); Gu=nx.convert_node_labels_to_integers(Gu)
    raw={i:frozenset(Gu.nodes[i]["community"]) for i in Gu.nodes()}; uq={c:j for j,c in enumerate(set(raw.values()))}
    comm={i:uq[raw[i]] for i in Gu.nodes()}
    return DS(_orient(Gu,recip,rng),_lab(n,priv,rng),comm,list(range(n)),_feat(n,len(uq),comm,homo,fd,rng))
def norm_adj(G,n):
    und=G.to_undirected(); edges=list(und.edges())
    A=torch.zeros((n,n))
    if edges:
        eu,ev=zip(*edges); eu=torch.tensor(eu,dtype=torch.long); ev=torch.tensor(ev,dtype=torch.long)
        A[eu,ev]=1.0; A[ev,eu]=1.0
    A+=torch.eye(n); deg=A.sum(1); di=torch.pow(deg,-0.5); di[torch.isinf(di)]=0
    return (A*di.unsqueeze(1)*di.unsqueeze(0)).to(DEVICE)
class LightGCN(nn.Module):
    def __init__(s,n,dim=64,layers=3):
        super().__init__(); s.emb=nn.Embedding(n,dim); nn.init.normal_(s.emb.weight,std=0.1); s.layers=layers
    def forward(s,Ah):
        e=s.emb.weight; outs=[e]
        for _ in range(s.layers): e=Ah@e; outs.append(e)
        return torch.stack(outs,0).mean(0)
def train(model,Ah,pos,n,epochs=30,bs=1024,lr=0.01):
    model = model.to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-5)
    pos=torch.tensor(pos,dtype=torch.long,device=DEVICE)
    for ep in range(epochs):
        perm=torch.randperm(pos.size(0),device=DEVICE)
        for i in range(0,pos.size(0),bs):
            idx=perm[i:i+bs]; u,p=pos[idx,0],pos[idx,1]; neg=torch.randint(0,n,(u.size(0),),device=DEVICE)
            E=model(Ah); su=(E[u]*E[p]).sum(1); sn=(E[u]*E[neg]).sum(1)
            loss=-torch.log(torch.sigmoid(su-sn)+1e-9).mean(); opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad(): return model(Ah).detach().cpu().numpy()

## 3. Defenses + fixed/adaptive attacker

- **Noise(sig):** N(0,sig) is added to the score. Fixed = single draw (Q=1). Adaptive = average of Q draws (the optimal estimator for zero-mean noise -> as Q grows, it approaches the undefended case).
- **Pool(p):** each displayed slot is replaced with a random node with probability p. Fixed = single list. Adaptive = frequency count across Q lists (true neighbors appear consistently).
- **Diversification(lam):** MMR re-ranking (low lam = aggressive). Fixed = diversified top-K. Adaptive = re-rank the length K'=b*K list by raw relevance and take the top K.
- **Rate-limit:** Q_max axis; adaptive leakage is read at Q=Q_max.

In [ ]:
def _prec_idx(idx,Nt,k): return len(set(list(idx)[:k])&Nt)/k
def _prec_top(scores,Nt,k):
    idx=np.argpartition(-scores,k)[:k]; return len(set(idx.tolist())&Nt)/k
def _mmr(s,sims,pool_idx,k,lam):
    rem=list(range(len(pool_idx))); sel=[]; mx=np.full(len(pool_idx),-1e9)
    while len(sel)<k and rem:
        if not sel: bi=max(rem,key=lambda i:s[pool_idx[i]])
        else: bi=max(rem,key=lambda i:lam*s[pool_idx[i]]-(1-lam)*mx[i])
        sel.append(bi); rem.remove(bi); mx=np.maximum(mx,sims[:,bi])
    return [pool_idx[i] for i in sel]

# ---- Real (e,d)-DP helpers (same math as in 04; see rationale there) ----
def _delta_of_sigma_eps(sigma, eps, sensitivity):
    a = sensitivity/(2*sigma) - eps*sigma/sensitivity
    b = -sensitivity/(2*sigma) - eps*sigma/sensitivity
    return norm.cdf(a) - math.exp(eps + norm.logcdf(b))
def gaussian_sigma_for_eps(eps, delta, sensitivity):
    f = lambda s: _delta_of_sigma_eps(s, eps, sensitivity) - delta
    lo, hi = 1e-8, max(sensitivity, 1.0)
    while f(hi) > 0: hi *= 2
    while f(lo) < 0: lo /= 2
    return brentq(f, lo, hi, xtol=1e-12, rtol=1e-12)
def rdp_rho(sigma, sensitivity): return (sensitivity**2)/(2*sigma**2)
def zcdp_to_eps(rho, delta): return rho + 2*math.sqrt(rho*math.log(1/delta))
def cumulative_eps(eps_step, delta, sensitivity, Q):
    sigma = gaussian_sigma_for_eps(eps_step, delta, sensitivity)
    return zcdp_to_eps(Q*rdp_rho(sigma, sensitivity), delta), sigma

def evaluate(E,ds,seed=0,k=K):
    En=E/(np.linalg.norm(E,axis=1,keepdims=True)+1e-9); rng=np.random.default_rng(seed+777)
    n=En.shape[0]; targets=[t for t in ds.node_list if ds.private[t] and ds.G.degree(t)>=4]
    maxQ=max(Qs); acc={}
    def push(kk,v): acc.setdefault(kk,[]).append(v)
    Delta2_dp = 2*DP_CLIP*np.sqrt(n)
    for t in targets:
        Nt=ds.neigh(t)
        if not Nt: continue
        s=En@En[t]; s[t]=-1e9
        push("undef",_prec_top(s,Nt,k))
        # noise
        for sig in SIGMAS:
            running=np.zeros(n)
            for q in range(1,maxQ+1):
                running+=s+rng.normal(0,sig,n)
                if q==1: push(f"noise_static|sig={sig}",_prec_top(running.copy(),Nt,k))
                if q in Qs: push(f"noise_adapt|sig={sig}|Q={q}",_prec_top(running/q,Nt,k))
        # differential privacy (Reviewer 3, item 5): calibrated Gaussian mechanism, SAME Q-aggregation scenario.
        # clip is first done over the RAW `s`, but the self-suggestion mask (t) is AFTER the clip
        # is reapplied -- otherwise -1e9 would rise to -1 via clip and leak into self-recommendation.
        s_clip = np.clip(s, -DP_CLIP, DP_CLIP); s_clip[t] = -1e9
        for eps in EPS_STEPS:
            sigma_dp = gaussian_sigma_for_eps(eps, DP_DELTA, Delta2_dp)
            running=np.zeros(n)
            for q in range(1,maxQ+1):
                running+=s_clip+rng.normal(0,sigma_dp,n)
                if q==1: push(f"dp_static|eps={eps}",_prec_top(running.copy(),Nt,k))
                if q in Qs: push(f"dp_adapt|eps={eps}|Q={q}",_prec_top(running/q,Nt,k))
        # pool dilution
        base=np.argsort(-s)[:maxQ*k]
        for p in POOLS:
            tally=np.zeros(n)
            for q in range(1,maxQ+1):
                disp=[ (int(rng.integers(n)) if rng.random()<p else int(base[r])) for r in range(k) ]
                if q==1: push(f"pool_static|p={p}",_prec_idx(disp,Nt,k))
                for d in disp: tally[d]+=1
                if q in Qs: push(f"pool_adapt|p={p}|Q={q}",len(set(np.argsort(-tally)[:k].tolist())&Nt)/k)
        # diversification
        pool_idx=np.argsort(-s)[:POOL_M]; sims=(En[pool_idx]@En[pool_idx].T)
        for lam in LAMS:
            ml=_mmr(s,sims,pool_idx,max(DIVBUDG)*k,lam)
            push(f"div_static|lam={lam}",_prec_idx(ml,Nt,k))
            for b in DIVBUDG:
                sub=sorted(ml[:b*k],key=lambda c:-s[c])[:k]
                push(f"div_adapt|lam={lam}|b={b}",len(set(sub)&Nt)/k)
    return {kk:float(np.mean(v)) for kk,v in acc.items()}, len(acc["undef"])

## 4. ETA and the multi-seed run

In [ ]:
def run_seed(gt,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    ds=(make_sbm(n=N_NODES,seed=seed,**BASELINE) if gt=="sbm" else make_lfr(n=N_NODES,seed=seed,homo=BASELINE["homo"],priv=BASELINE["priv"],recip=BASELINE["recip"]))
    if ds is None: return None,0
    n=len(ds.node_list); Ah=norm_adj(ds.G,n); pos=[(u,v) for u,v in ds.G.to_undirected().edges()]
    E=train(LightGCN(n),Ah,pos,n,epochs=EPOCHS)
    return evaluate(E,ds,seed=seed)

_t=time.time(); _r,_=run_seed(GRAPHS[0],SEEDS[0]); per=time.time()-_t
total=len(SEEDS)*len(GRAPHS)
print(f"~{per:.1f}s/seed | {total} seed-runs | estimated ~{per*total/60:.1f} min")

In [ ]:
rows=[]; t0=time.time(); done=0
sig=NOM["sig"]
for gt in GRAPHS:
    for seed in SEEDS:
        res,nt=run_seed(gt,seed); done+=1
        if res is None: print(f"[atlandi] {gt} seed={seed}"); continue
        for kk,v in res.items(): rows.append(dict(graph=gt,seed=seed,condition=kk,value=v,n_targets=nt))
        pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR,"adaptive_long.csv"),index=False)
        st=res[f"noise_static|sig={sig}"]; ad=res[f"noise_adapt|sig={sig}|Q=40"]
        print(f"[{done}/{total}] {gt} seed={seed} undef={res['undef']:.3f} | noise sig={sig}: static={st:.3f} -> adaptiveQ40={ad:.3f}")
df=pd.DataFrame(rows); print(f"\nBitti: {len(df)} satir, {time.time()-t0:.1f}s")

## 5. Aggregate and write the run summary

In [ ]:
def ci95(x):
    x=np.asarray(x,float); n=len(x)
    return 1.96*x.std(ddof=1)/math.sqrt(n) if n>1 else 0.0
df=pd.read_csv(os.path.join(OUT_DIR,"adaptive_long.csv"))
def agg(cond,graph):
    v=df[(df.condition==cond)&(df.graph==graph)]["value"].values
    return (float(np.mean(v)),ci95(v)) if len(v) else (float('nan'),float('nan'))
graphs=sorted(df.graph.unique()); sig=NOM["sig"]; p=NOM["p"]; lam=NOM["lam"]; eps_nom=NOM["eps"]; Qm=max(Qs); bm=max(DIVBUDG)
L=["### 1) Adaptivity gap - defense at nominal point (cold prec@20, mu+-CI; across seeds)",
   "(static = fixed attacker; adaptive = best-response full budget; residual = undef - adaptive)","",
   "| graph | defense | knob | undef | static | adaptive | gap (ad-st) | residual protection (undef-ad) |",
   "|---|---|---|---|---|---|---|---|"]
for g in graphs:
    u=agg("undef",g)[0]
    for name,stat,adp,knob in [
        ("noise",f"noise_static|sig={sig}",f"noise_adapt|sig={sig}|Q={Qm}",f"sig={sig},Q={Qm}"),
        ("dp (genuine DP)",f"dp_static|eps={eps_nom}",f"dp_adapt|eps={eps_nom}|Q={Qm}",f"eps={eps_nom},Q={Qm}"),
        ("pool", f"pool_static|p={p}",     f"pool_adapt|p={p}|Q={Qm}",     f"p={p},Q={Qm}"),
        ("diversify",f"div_static|lam={lam}",f"div_adapt|lam={lam}|b={bm}",f"lam={lam},x{bm}")]:
        st=agg(stat,g); ad=agg(adp,g)
        L.append(f"| {g} | {name} | {knob} | {u:.3f} | {st[0]:.3f}+-{st[1]:.3f} | {ad[0]:.3f}+-{ad[1]:.3f} | "
                 f"{ad[0]-st[0]:+.3f} | {u-ad[0]:.3f} |")
L+=["","### 2) Noise x rate-limit (cold prec@20 mean): adaptive leakage is capped at Q_max","",
    "| graph | sigma | "+" | ".join(f"Q_max={q}" for q in Qs)+" |",
    "|---|---|"+"|".join(["---"]*len(Qs))+"|"]
for g in graphs:
    for s_ in SIGMAS:
        L.append(f"| {g} | {s_} | "+" | ".join(f"{agg(f'noise_adapt|sig={s_}|Q={q}',g)[0]:.3f}" for q in Qs)+" |")
L+=["","### 3) (NEW) Real DP x rate-limit: cold prec@20 mu AND spent cumulative epsilon (zCDP)","",
    "(Referee 3.5 -- no longer an ad hoc 'feeling': for each Q_max the epsilon ACTUALLY spent can be traced with the formula.)","",
    "| graph | eps_step | "+" | ".join(f"Q_max={q}" for q in Qs)+" |",
    "|---|---|"+"|".join(["---"]*len(Qs))+"|"]
for g in graphs:
    for e_ in EPS_STEPS:
        L.append(f"| {g} | {e_} | "+" | ".join(f"{agg(f'dp_adapt|eps={e_}|Q={q}',g)[0]:.3f}" for q in Qs)+" |")
Delta2_rep = 2*DP_CLIP*np.sqrt(N_NODES)
L += ["", "*cumulative epsilon (zCDP, independent of graph size: a function of eps_step and Q alone; delta="+str(DP_DELTA)+"):*", "",
      "| eps_step | "+" | ".join(f"Q_max={q}" for q in Qs)+" |", "|---|"+"|".join(["---"]*len(Qs))+"|"]
for e_ in EPS_STEPS:
    L.append(f"| {e_} | "+" | ".join(f"{cumulative_eps(e_, DP_DELTA, Delta2_rep, q)[0]:.1f}" for q in Qs)+" |")
L+=["","### 4) Diversification: recovery with an adaptive budget (cold prec@20 mean)","",
    "| graph | lam | static |"+"".join(f" x{b} |" for b in DIVBUDG),
    "|---|---|---|"+"|".join(["---"]*len(DIVBUDG))+"|"]
for g in graphs:
    for l_ in LAMS:
        L.append(f"| {g} | {l_} | {agg(f'div_static|lam={l_}',g)[0]:.3f} |"+
                 "".join(f" {agg(f'div_adapt|lam={l_}|b={b}',g)[0]:.3f} |" for b in DIVBUDG))
meta=[f"# Cold-Start: Adaptive Attacker summary","",
      f"- MODE={MODE} | K={K} | seeds={len(SEEDS)} | graphs={graphs} | epochs={EPOCHS} | device={DEVICE}",
      f"- undef cold prec@20: "+", ".join(f"{g}={agg('undef',g)[0]:.3f}" for g in graphs),""]
share="\n".join(meta+L)+"\n"
open(os.path.join(OUT_DIR,"RUN_SUMMARY.md"),"w").write(share)
print("="*70); print(share); print("="*70)
print("The block between the two ===== lines is the run summary.")

## 6. (Optional) Three-panel figure

In [ ]:
import matplotlib.pyplot as plt
df=pd.read_csv(os.path.join(OUT_DIR,"adaptive_long.csv")); g=GRAPHS[0]
def mean_of(cond):
    v=df[(df.condition==cond)&(df.graph==g)]["value"].values; return float(np.mean(v)) if len(v) else np.nan
undef=mean_of("undef")
fig,ax=plt.subplots(1,3,figsize=(15,4))
for s_ in SIGMAS:
    ax[0].plot(Qs,[mean_of(f"noise_adapt|sig={s_}|Q={q}") for q in Qs],marker="o",label=f"sig={s_}")
ax[0].axhline(undef,ls="--",c="k",label="undefended"); ax[0].set_xscale("log")
ax[0].set_xlabel("attacker query budget Q (= rate-limit Q_max)"); ax[0].set_ylabel("cold prec@20")
ax[0].set_title(f"Output noise: adaptive recovery ({g})"); ax[0].legend()
for l_ in LAMS:
    ax[1].plot([0]+list(DIVBUDG),[mean_of(f"div_static|lam={l_}")]+[mean_of(f"div_adapt|lam={l_}|b={b}") for b in DIVBUDG],marker="s",label=f"lam={l_}")
ax[1].axhline(undef,ls="--",c="k"); ax[1].set_xlabel("adaptive budget multiplier (0=static)"); ax[1].set_ylabel("cold prec@20")
ax[1].set_title("Diversification: recovery with budget"); ax[1].legend()
names=["noise","pool","diversify"]; sig=NOM["sig"]; p=NOM["p"]; lam=NOM["lam"]; Qm=max(Qs); bm=max(DIVBUDG)
stat=[mean_of(f"noise_static|sig={sig}"),mean_of(f"pool_static|p={p}"),mean_of(f"div_static|lam={lam}")]
adp=[mean_of(f"noise_adapt|sig={sig}|Q={Qm}"),mean_of(f"pool_adapt|p={p}|Q={Qm}"),mean_of(f"div_adapt|lam={lam}|b={bm}")]
x=np.arange(3); w=0.35
ax[2].bar(x-w/2,stat,w,label="static"); ax[2].bar(x+w/2,adp,w,label="adaptive")
ax[2].axhline(undef,ls="--",c="k",label="undefended"); ax[2].set_xticks(x); ax[2].set_xticklabels(names)
ax[2].set_ylabel("cold prec@20"); ax[2].set_title("Adaptivity gap"); ax[2].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR,"adaptive_figs.png"),dpi=130); plt.show()
print("[saved]",os.path.join(OUT_DIR,"adaptive_figs.png"))